In [1]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
# -------------------------------
# 1. Génération de données factices (série temporelle)
# -------------------------------
np.random.seed(0)
t = np.arange(0, 200, 0.1)
data = np.sin(t) + 0.1*np.random.randn(len(t))

# Normalisation
scaler = MinMaxScaler(feature_range=(0, 1))
data = scaler.fit_transform(data.reshape(-1, 1))

# -------------------------------
# 2. Création des séquences
# -------------------------------
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

seq_length = 20
X, y = create_sequences(data, seq_length)

# Train / Test split
train_size = int(len(X)*0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Conversion en tenseurs
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)
X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test)

# -------------------------------
# 3. Modèle LSTM
# -------------------------------
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]   # dernière sortie
        out = self.fc(out)
        return out

# -------------------------------
# 4. Modèle GRU
# -------------------------------
class GRUModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=2):
        super(GRUModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.gru(x, h0)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

# -------------------------------
# 5. Entraînement générique
# -------------------------------
def train_model(model, X_train, y_train, epochs=20, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        
        if (epoch+1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.6f}")

# -------------------------------
# 6. Évaluation avec RMSE, MAE, MAPE
# -------------------------------
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        predictions = model(X_test).numpy()
        y_true = y_test.numpy()

    # Inversion de la normalisation
    predictions = scaler.inverse_transform(predictions)
    y_true = scaler.inverse_transform(y_true)

    rmse = np.sqrt(mean_squared_error(y_true, predictions))
    mae = mean_absolute_error(y_true, predictions)
    mape = np.mean(np.abs((y_true - predictions) / y_true)) * 100

    return rmse, mae, mape

# -------------------------------
# 7. Entraîner et tester LSTM
# -------------------------------
print("===== LSTM =====")
lstm_model = LSTMModel()
train_model(lstm_model, X_train, y_train, epochs=20)

rmse, mae, mape = evaluate_model(lstm_model, X_test, y_test)
print(f"LSTM -> RMSE: {rmse:.4f}, MAE: {mae:.4f}, MAPE: {mape:.2f}%")

# -------------------------------
# 8. Entraîner et tester GRU
# -------------------------------
print("\n===== GRU =====")
gru_model = GRUModel()
train_model(gru_model, X_train, y_train, epochs=20)

rmse, mae, mape = evaluate_model(gru_model, X_test, y_test)
print(f"GRU -> RMSE: {rmse:.4f}, MAE: {mae:.4f}, MAPE: {mape:.2f}%")

===== LSTM =====
Epoch [5/20], Loss: 0.241471
Epoch [10/20], Loss: 0.158228
Epoch [15/20], Loss: 0.080152
Epoch [20/20], Loss: 0.084915
LSTM -> RMSE: 0.7480, MAE: 0.6291, MAPE: 271.16%

===== GRU =====
Epoch [5/20], Loss: 0.167961
Epoch [10/20], Loss: 0.076505
Epoch [15/20], Loss: 0.072958
Epoch [20/20], Loss: 0.065285
GRU -> RMSE: 0.6313, MAE: 0.5483, MAPE: 174.68%


### Ce code montre tout le pipeline : ### 

Création et normalisation des données

Construction des séquences temporelles

Définition des modèles LSTM et GRU

Entraînement

Évaluation avec :

1. RMSE → pénalise fortement les grandes erreurs

2. MAE → robuste aux outliers

3. MAPE → interprétation en pourcentage

C’est exactement le type d’exemple « complet » qu’on trouve dans les travaux de prévision de séries temporelles avec LSTM/GRU.